# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Data Joins and JSON columns** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [ ]:
from pcamarillor.spark_utils import SparkUtils

# Create Spark session
su = SparkUtils("Example - Extract info from JSON", "spark://spark-master:7077")

# Manipulating JSON Columns

In [ ]:
json_schema = SparkUtils.generate_schema([("id", "int"), ("json_col", "string")])
json_data = [
    (1, '{"name": "Alice", "age": 25, "payments": [34, 433, 54], "address": {"city": "New York", "zip": "10001"}}'),
    (2, '{"name": "Bob", "age": 30, "address": {"city": "Los Angeles", "zip": "90001"}}'),
    (3, '{"name": "Charlie", "age": 35, "address": {"city": "Chicago", "zip": "60601"}}')
]

df_json = su._spark.createDataFrame(json_data, json_schema)
df_json.show(truncate=False)


### Extract a JSON column with get_json_object function

In [ ]:
from pyspark.sql.functions import get_json_object
df_json = df_json.withColumn("name", get_json_object(df_json.json_col, "$.name"))
df_json.show()

In [ ]:
df_json = df_json.withColumn("age", get_json_object(df_json.json_col, "$.age"))
df_json.show()

### Extact a JSON column with from_json

In [ ]:
from pyspark.sql.functions import from_json
# Deine the schema of the JSON object
people_schema = SparkUtils.generate_schema([("name", "string"),
                                            ("age", "int"),
                                            ("payments", "array_int"),
                                            ("address", "struct")])
df_parsed = df_json.withColumn("parsed", from_json(df_json.json_col, people_schema))
df_parsed.printSchema()
df_parsed.show(truncate=False)

In [ ]:
df_parsed.printSchema()

In [ ]:
from pyspark.sql.functions import col
df_parsed.select(col("parsed.name"), col("parsed.payments").getItem(1)).show()

## Data Join

In [ ]:
df_a = su._spark.createDataFrame([(1, "Alice"), (2, "Bob")],
                              ["id", "name"])
df_b = su._spark.createDataFrame([(3, "Charlie"), (4, "David")],
                              ["id", "name"])
result = df_a.union(df_b)
result.show()

### Union w/o duplicates

In [ ]:
df_a = su._spark.createDataFrame([(1, "Alice"), (2, "Bob")],
                              ["id", "name"])
df_b = su._spark.createDataFrame([(1, "Alice"), (4, "David")],
                              ["id", "name"])
df_a.union(df_b).distinct().show()

### Union with Mismatched Schemas

In [ ]:
df_a = su._spark.createDataFrame([(1, "Alice")], ["id", "name"])
df_b = su._spark.createDataFrame([("Bob", 2)], ["name", "id"])
result = df_a.unionByName(df_b)
result.show()

### Union by Name with missing columns

In [ ]:
df_a = su._spark.createDataFrame([(1, "Alice", "NY")], ["id", "name", "city"])
df_a.show()
df_b = su._spark.createDataFrame([(2, "Bob")], ["id", "name"])
df_b.show()
result = df_a.unionByName(df_b, allowMissingColumns=True)
result.show()

## Left Join

### Datasets

In [ ]:
book_data = [
    ("Game of thrones", 400, 1),
    ("Spark", 500, 2),
    ("Kafka", 300, 3),
    ("Java", 350, 5)
]
df_books = su._spark.createDataFrame(book_data, ["book_name", "cost", "writer_id"])

writer_data = [
    ("George R.R. Martin", 1),
    ("Zaharia", 2),
    ("Neha", 3),
    ("James", 4)
]
df_writers = su._spark.createDataFrame(writer_data, ["writer_name", "writer_id"])

df_books.show()
df_writers.show()

In [ ]:
result = df_books.join(df_writers, 
      df_books["writer_id"] == df_writers["writer_id"], "left")
result.show()

In [ ]:
result = df_books.join(df_writers, on="writer_id", how="left")
result.show()

## Right join

In [ ]:
result = df_books.join(df_writers, df_books["writer_id"] == df_writers["writer_id"], "right")
result.show()

In [ ]:
result = df_books.join(df_writers, on="writer_id", how="right")
result.show()

### Inner Join

In [ ]:
df_books.join(df_writers, on="writer_id").show()

In [ ]:
su._spark.stop()